# Módulo IV – Aula 2: execução individual
**Gustavo Carvalho de Oliveira | UDF | Profa. Kadidja Valéria | 2026.2**

Execução própria do laboratório supervisionado, a partir do notebook didático. Diferença em relação ao roteiro base: `max_depth` **e** `min_samples_leaf` são escolhidos juntos por validação cruzada **antes** de qualquer uso do teste. Todos os dados são **sintéticos**.

In [1]:
import sys, json
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             mean_absolute_error, r2_score, recall_score, precision_score)
SEMENTE = 42
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| matplotlib:', matplotlib.__version__, '| scikit-learn:', sklearn.__version__)

Python: 3.12.3
NumPy: 2.5.4 | pandas: 3.0.6 | matplotlib: 3.11.2 | scikit-learn: 1.9.1


## 1. Dados (mesma simulação do material didático)

In [2]:
rng = np.random.default_rng(SEMENTE)
n = 600
frequencia = rng.uniform(35, 100, n); nota = rng.uniform(0, 10, n); entregas = rng.integers(0, 9, n)
logit = -0.4 + (65 - frequencia)/18 + (5 - nota)/2 + (4 - entregas)/3
alvo = rng.binomial(1, 1/(1+np.exp(-logit)))
dados = pd.DataFrame({'frequencia_sem4': frequencia, 'nota_sem4': nota,
                      'entregas_sem4': entregas, 'evasao_fim_semestre': alvo})
dados.loc[rng.choice(n, 30, replace=False), 'nota_sem4'] = np.nan
csv = pd.read_csv('../dados/educacionais-sinteticos.csv')
print('Igual ao CSV fornecido:', np.allclose(csv.fillna(-1).values, dados.fillna(-1).values))
print('Ausentes:', dados.isna().sum().to_dict())
print('Classes (%):', dados['evasao_fim_semestre'].value_counts(normalize=True).mul(100).round(1).to_dict())

Igual ao CSV fornecido: True
Ausentes: {'frequencia_sem4': 0, 'nota_sem4': 30, 'entregas_sem4': 0, 'evasao_fim_semestre': 0}
Classes (%): {0: 59.0, 1: 41.0}


## 2. Divisão antes de qualquer transformação

In [3]:
atributos = ['frequencia_sem4', 'nota_sem4', 'entregas_sem4']
X, y = dados[atributos], dados['evasao_fim_semestre']
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMENTE)
print('Treino:', len(X_treino), '| Teste:', len(X_teste))

Treino: 450 | Teste: 150


## 3. Seleção conjunta de profundidade e folha (só treino)
Critério: maior acurácia média de validação; em empate dentro de 0,005, fica a configuração mais simples (menor profundidade, maior folha).

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
def arvore(prof, folha, **kw):
    return Pipeline([('imputador', SimpleImputer(strategy='median')),
                     ('arvore', DecisionTreeClassifier(max_depth=prof, min_samples_leaf=folha, random_state=SEMENTE, **kw))])
linhas = []
for prof in [2, 3, 5, None]:
    for folha in [5, 10, 20, 40]:
        s = cross_validate(arvore(prof, folha), X_treino, y_treino, cv=cv, scoring='accuracy', return_train_score=True)
        linhas.append({'prof': str(prof), 'folha': folha, 'treino': s['train_score'].mean(),
                       'validacao': s['test_score'].mean(), 'desvio': s['test_score'].std()})
grade = pd.DataFrame(linhas)
print(grade.round(4).to_string(index=False))
melhor_val = grade['validacao'].max()
cand = grade[grade['validacao'] >= melhor_val - 0.005].copy()
cand['prof_num'] = cand['prof'].replace('None', '99').astype(int)
esc = cand.sort_values(['prof_num', 'folha'], ascending=[True, False]).iloc[0]
PROF = None if esc['prof'] == 'None' else int(esc['prof']); FOLHA = int(esc['folha'])
print('\nEscolhida no treino: max_depth =', PROF, '| min_samples_leaf =', FOLHA)
print(esc[['treino','validacao','desvio']].astype(float).round(4).to_dict())
modelo = arvore(PROF, FOLHA).fit(X_treino, y_treino)

prof  folha  treino  validacao  desvio
   2      5  0.7328     0.6844  0.0425
   2     10  0.7328     0.6844  0.0425
   2     20  0.7328     0.6844  0.0425
   2     40  0.7328     0.6844  0.0425
   3      5  0.7706     0.7000  0.0233
   3     10  0.7678     0.7022  0.0257
   3     20  0.7656     0.6978  0.0325
   3     40  0.7661     0.7111  0.0351
   5      5  0.8261     0.7222  0.0298
   5     10  0.8111     0.7222  0.0391
   5     20  0.7900     0.7267  0.0480
   5     40  0.7756     0.7289  0.0303
None      5  0.8678     0.7133  0.0215
None     10  0.8206     0.7111  0.0477
None     20  0.7900     0.7267  0.0480
None     40  0.7756     0.7289  0.0303

Escolhida no treino: max_depth = 5 | min_samples_leaf = 40
{'treino': 0.7756, 'validacao': 0.7289, 'desvio': 0.0303}


## 4. Teste final (usado uma única vez)

In [5]:
prev = modelo.predict(X_teste)
base = DummyClassifier(strategy='most_frequent').fit(X_treino, y_treino)
acc, acc_base = accuracy_score(y_teste, prev), accuracy_score(y_teste, base.predict(X_teste))
cm = confusion_matrix(y_teste, prev, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()
print(f'Acurácia treino completo: {modelo.score(X_treino, y_treino):.4f}')
print(f'Acurácia teste: {acc:.4f} | referência: {acc_base:.4f}')
print('TN', tn, 'FP', fp, 'FN', fn, 'TP', tp)
print(f'Sensibilidade: {tp/(tp+fn):.4f} | Precisão: {tp/(tp+fp):.4f}')
imp = dict(zip(atributos, modelo.named_steps['arvore'].feature_importances_.round(3))); print('Importâncias:', imp)
t = modelo.named_steps['arvore'].tree_
print('Raiz:', atributos[t.feature[0]], '<=', round(t.threshold[0], 2))
for lado, no in [('esq', t.children_left[0]), ('dir', t.children_right[0])]:
    if t.feature[no] >= 0: print(lado, atributos[t.feature[no]], '<=', round(t.threshold[no], 2))
ConfusionMatrixDisplay(cm, display_labels=['Permaneceu (0)', 'Evadiu (1)']).plot(cmap='Greens')
plt.title('Teste final — dados sintéticos'); plt.tight_layout(); plt.savefig('../assets/matriz-confusao.png', dpi=120); plt.close()
plt.figure(figsize=(14, 7))
plot_tree(modelo.named_steps['arvore'], feature_names=atributos, class_names=['Permaneceu', 'Evadiu'], max_depth=2, fontsize=9)
plt.title('Primeiros níveis da árvore'); plt.tight_layout(); plt.savefig('../assets/arvore.png', dpi=110); plt.close()

Acurácia treino completo: 0.7622
Acurácia teste: 0.7800 | referência: 0.5933
TN 70 FP 19 FN 14 TP 47
Sensibilidade: 0.7705 | Precisão: 0.7121
Importâncias: {'frequencia_sem4': np.float64(0.21), 'nota_sem4': np.float64(0.7), 'entregas_sem4': np.float64(0.09)}
Raiz: nota_sem4 <= 6.21
esq frequencia_sem4 <= 64.4
dir frequencia_sem4 <= 65.5


## 5. Experimentos após o teste (somente validação cruzada no treino)
Qualquer mudança adotada a partir daqui exigiria um novo teste independente.

In [6]:
def cv_metr(m, Xt=X_treino):
    s = cross_validate(m, Xt, y_treino, cv=cv, scoring=['accuracy', 'recall', 'precision'])
    return {k: (s['test_'+k].mean().round(4), s['test_'+k].std().round(4)) for k in ['accuracy', 'recall', 'precision']}
print('Exp. 1 — pesos de classe')
print('  padrão  :', cv_metr(arvore(PROF, FOLHA)))
print('  balanced:', cv_metr(arvore(PROF, FOLHA, class_weight='balanced')))
print('Exp. 2 — só um atributo por vez')
for a in atributos:
    print(' ', a, cv_metr(arvore(PROF, FOLHA), X_treino[[a]]))

Exp. 1 — pesos de classe
  padrão  : {'accuracy': (np.float64(0.7289), np.float64(0.0303)), 'recall': (np.float64(0.6054), np.float64(0.1297)), 'precision': (np.float64(0.6995), np.float64(0.0346))}
  balanced: {'accuracy': (np.float64(0.7133), np.float64(0.031)), 'recall': (np.float64(0.6919), np.float64(0.0991)), 'precision': (np.float64(0.642), np.float64(0.0365))}
Exp. 2 — só um atributo por vez
  frequencia_sem4 {'accuracy': (np.float64(0.5956), np.float64(0.0327)), 'recall': (np.float64(0.3676), np.float64(0.0501)), 'precision': (np.float64(0.5111), np.float64(0.0571))}


  nota_sem4 {'accuracy': (np.float64(0.6756), np.float64(0.0267)), 'recall': (np.float64(0.5297), np.float64(0.0946)), 'precision': (np.float64(0.6305), np.float64(0.0489))}
  entregas_sem4 {'accuracy': (np.float64(0.5867), np.float64(0.0537)), 'recall': (np.float64(0.4), np.float64(0.0844)), 'precision': (np.float64(0.5044), np.float64(0.0943))}


## 6. Regressão — biblioteca (seção independente)

In [7]:
rng_reg = np.random.default_rng(SEMENTE)
reg = pd.DataFrame({'usuarios_ativos': rng_reg.uniform(100, 1000, 300), 'horas_abertura': rng_reg.uniform(20, 60, 300)})
reg['emprestimos_semana'] = 30 + 0.35*reg['usuarios_ativos'] + 2.5*reg['horas_abertura'] + rng_reg.normal(0, 35, 300)
Xr, yr = reg[['usuarios_ativos', 'horas_abertura']], reg['emprestimos_semana']
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=SEMENTE)
lin = lambda: Pipeline([('imputador', SimpleImputer(strategy='median')), ('linear', LinearRegression())])
regressor = lin().fit(Xr_tr, yr_tr); pr = regressor.predict(Xr_te)
ref = DummyRegressor(strategy='mean').fit(Xr_tr, yr_tr)
print(f'MAE: {mean_absolute_error(yr_te, pr):.2f} | MAE ref: {mean_absolute_error(yr_te, ref.predict(Xr_te)):.2f} | R2: {r2_score(yr_te, pr):.4f}')
print('Coef:', regressor.named_steps['linear'].coef_.round(3), 'Intercepto:', round(regressor.named_steps['linear'].intercept_, 2))
print(pd.DataFrame({'real': yr_te, 'previsto': pr, 'erro_abs': np.abs(yr_te-pr)}).head().round(2).to_string())
kf = KFold(5, shuffle=True, random_state=SEMENTE)
for cols in [['usuarios_ativos', 'horas_abertura'], ['usuarios_ativos'], ['horas_abertura']]:
    s = cross_validate(lin(), Xr_tr[cols], yr_tr, cv=kf, scoring='neg_mean_absolute_error')
    print('CV MAE', cols, round(-s['test_score'].mean(), 2), '±', round(s['test_score'].std(), 2))
res = yr_te - pr
print('Resíduos: média', round(res.mean(), 2), 'desvio', round(res.std(), 2), 'min', round(res.min(), 1), 'max', round(res.max(), 1))
print('Previsão 500 usuários/40 h:', round(regressor.predict(pd.DataFrame({'usuarios_ativos': [500], 'horas_abertura': [40]}))[0], 1))
plt.figure(figsize=(6, 5)); plt.scatter(yr_te, pr, alpha=0.7, color='#2e7d32')
lim = [min(yr_te.min(), pr.min()), max(yr_te.max(), pr.max())]; plt.plot(lim, lim, 'k--', label='Previsto = observado')
plt.xlabel('Observado'); plt.ylabel('Previsto'); plt.title('Regressão no teste'); plt.legend(); plt.tight_layout()
plt.savefig('../assets/regressao-observado-previsto.png', dpi=120); plt.close()
plt.figure(figsize=(6, 4)); plt.scatter(pr, res, color='#2e7d32'); plt.axhline(0, color='k', ls='--')
plt.xlabel('Previsto'); plt.ylabel('Resíduo'); plt.title('Resíduos no teste'); plt.tight_layout()
plt.savefig('../assets/regressao-residuos.png', dpi=120); plt.close()
json.dump({'prof': PROF, 'folha': FOLHA, 'acc_teste': float(acc), 'acc_ref': float(acc_base), 'matriz': cm.tolist(),
           'mae': float(mean_absolute_error(yr_te, pr)), 'r2': float(r2_score(yr_te, pr)), 'sklearn': sklearn.__version__},
          open('resultados_execucao.json', 'w'), indent=2)

MAE: 28.59 | MAE ref: 84.73 | R2: 0.8774
Coef: [0.348 2.297] Intercepto: 37.32
       real  previsto  erro_abs
203  208.90    209.40      0.50
266  277.71    277.32      0.39
152  190.95    227.18     36.23
9    229.67    271.95     42.29
233  359.77    363.62      3.85
CV MAE ['usuarios_ativos', 'horas_abertura'] 28.89 ± 1.77
CV MAE ['usuarios_ativos'] 35.9 ± 3.19
CV MAE ['horas_abertura'] 83.73 ± 5.36
Resíduos: média -1.28 desvio 35.47 min -87.2 max 64.6
Previsão 500 usuários/40 h: 303.2
